In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [19]:
url = "https://malegislature.gov/Legislators/Members/House"
headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64)"}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

table = soup.find("table")
if not table:
    raise RuntimeError("No table found on page — check the selector or whether JS is required.")

thead = table.find("thead")
header_names = []
if thead:
    header_names = [th.get_text(strip=True) for th in thead.find_all("th")]

rows = table.select("tbody tr")
data = []

for row in rows:
    tds = row.find_all("td")
    offset = 0
    if len(tds) > 0:
        first_td = tds[0]
        if first_td.find("img") is not None or len(first_td.get_text(strip=True)) <= 1:
            offset = 1

    if header_names and len(header_names) >= 7:
        header_map = {h.lower(): i for i, h in enumerate(header_names)}

        def find_idx(keys):
            for k in keys:
                if k in header_map:
                    return header_map[k]
            return None

        idx_first = find_idx(["first name", "first"])
        idx_last  = find_idx(["last name", "last"])
        idx_dist  = find_idx(["district"])
        idx_party = find_idx(["party"])
        idx_room  = find_idx(["room"])
        idx_phone = find_idx(["phone number", "phone"])
        idx_email = find_idx(["email address", "email"])

        
        if None not in (idx_first, idx_last, idx_dist, idx_party, idx_room, idx_phone, idx_email):
            cols = [td.get_text(strip=True) for td in tds]
            def g(i):
                return cols[i] if 0 <= i < len(cols) else ""
            data.append({ "First Name": g(idx_first), "Last Name": g(idx_last),"District": g(idx_dist), "Party": g(idx_party),"Room": g(idx_room),
        "Phone": g(idx_phone),"Email": g(idx_email),})
            continue  
            
    cols = [td.get_text(strip=True) for td in tds]
    cols_after = cols[offset:]
    while len(cols_after) < 7:
        cols_after.append("")

    first, last, district, party, room, phone, email = cols_after[:7]

    if first and ',' in first and not last:
        parts = [p.strip() for p in first.split(',') if p.strip()]
        if len(parts) >= 2:
            first = parts[0]
            last = parts[1]

    data.append({"First Name": first,"Last Name": last,"District": district,"Party": party,"Room": room,"Phone": phone,"Email": email,})

df = pd.DataFrame(data)
df = df[~(df["First Name"].eq("") & df["Last Name"].eq(""))].reset_index(drop=True)

out_file = "ma_house_legislators.csv"
df.to_csv(out_file, index=False)
print(df.head(10))

          First Name     Last Name        District       Party   Room  \
0              James       Arciero   2nd Middlesex    Democrat    134   
1              James  Arena-DeRosa   8th Middlesex    Democrat    167   
2  Jennifer Balinsky        Armini       8th Essex    Democrat     21   
3            Shirley       Arriaga     8th Hampden    Democrat    237   
4              Brian          Ashe     2nd Hampden    Democrat    160   
5              Bruce         Ayers     1st Norfolk    Democrat  473-C   
6           Michelle        Badger    1st Plymouth    Democrat     33   
7          Christine        Barber  34th Middlesex    Democrat    167   
8               John       Barrett   1st Berkshire    Democrat    277   
9             Donald    Berthiaume   5th Worcester  Republican    548   

            Phone                           Email  
0  (617) 722-2400       James.Arciero@mahouse.gov  
1  (617) 722-2810  James.Arena-DeRosa@mahouse.gov  
2  (617) 722-2140     Jennifer.Armini@ma

In [28]:
roll = pd.read_csv("rollcalls_all_years.csv")
leg = pd.read_csv("ma_house_legislators.csv")
# Extract last name = last token
roll["LastNameClean"] = roll["Legislator Name"].str.strip().str.split().str[-1]
leg["LastNameClean"] = leg["Last Name"].str.strip()
roll["LastNameClean"] = roll["LastNameClean"].str.replace("-", "").str.lower()
leg["LastNameClean"] = leg["LastNameClean"].str.replace("-", "").str.lower()
merged = roll.merge(
    leg,
    on="LastNameClean",
    how="left",
    suffixes=("", "_leg")
)
merged["Full Name"] = (
    merged["First Name"].fillna("") + " " +
    merged["Last Name"].fillna("")
).str.strip()
merged.loc[merged["Full Name"] == "", "Full Name"] = merged["Legislator Name"]

merged["Legislator Name"] = merged["Full Name"]

merged = merged.drop(columns=[
    "Full Name",
    "First Name",
    "Last Name",
    "LastNameClean"
])

merged.to_csv("rollcalls_with_legislator_info.csv", index=False)